Advanced RAG Techniques

1. No LangChain, native way
2. Using an LLM to divide up chunks in a sensible way
3. Using the best chunk size and encoder from yesterday
4. The LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [10]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

MODEL = "gpt-4.1-nano"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path.cwd().parent / "knowledge-base"
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

In [11]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [12]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

# STEP 1

In [13]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [14]:
documents = fetch_documents()

Loaded 124 documents


### On to step 2

In [ ]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a travel company called Skynest.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A copilot chat assistant will use these chunks to answer questions about Skynest's travel services, including flights, hotels, cars, packages, destinations, and visa policies.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [16]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a travel company called Skynest.
The document is of type: cars
The document has been retrieved from: c:/Users/jeffy/project/New/skynest_rag/knowledge-base/cars/bangkok_rental_fleet.md

A copilot chat assistant will use these chunks to answer questions about Skynest's travel services, including flights, hotels, cars, packages, destinations, and visa policies.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 7 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of t

In [17]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [18]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a travel company called Skynest.\nThe document is of type: cars\nThe document has been retrieved from: c:/Users/jeffy/project/New/skynest_rag/knowledge-base/cars/bangkok_rental_fleet.md\n\nA copilot chat assistant will use these chunks to answer questions about Skynest's travel services, including flights, hotels, cars, packages, destinations, and visa policies.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 7 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a hea

In [19]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [21]:
process_document(documents[50])

[Result(page_content='Route Overview: Tokyo NRT to Bangkok BKK\n\nThis chunk provides a summary of the flight route from Tokyo Narita to Bangkok Suvarnabhumi, including key details like duration and fare options.\n\n# SkyNest Route: Tokyo Narita (NRT) → Bangkok Suvarnabhumi (BKK)\n\n---\n\n## Route Summary\n\n| Detail | Info |\n|---|---|\n| **Origin** | Tokyo Narita (NRT) |\n| **Destination** | Bangkok Suvarnabhumi (BKK) |\n| **Typical Duration** | 6h 00m |\n| **Economy Fares from** | $320 |\n| **Business Class from** | $1,050 |\n\n---', metadata={'source': 'c:/Users/jeffy/project/New/skynest_rag/knowledge-base/flights_routes/tokyo_bangkok.md', 'type': 'flights_routes'})]

In [22]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [23]:
chunks = create_chunks(documents)

100%|██████████| 124/124 [16:09<00:00,  7.82s/it] 


In [24]:
print(len(chunks))

360


### Step 3 - save the embeddings

In [25]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [26]:
create_embeddings(chunks)

Vectorstore created with 360 documents


In [27]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'cyan'][['cars', 'company', 'destinations', 'flights_policies', 'flights_routes', 'hotels', 'packages', 'visa_policies'].index(t)] for t in doc_types]

In [29]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [28]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

We are doing advance RAG techiques

1. Reranking - reorder the rank results
2. Query re-writing

In [30]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [47]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [49]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [50]:
question = "Which hotel has the Aquaventure Waterpark?"
chunks = fetch_context_unranked(question)

In [54]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

Introduction to...
Luxury Hotels i...
Premium Tier Ho...
Luxury Tier Hot...
Premium Hotels ...
Hotels in Londo...
Luxury Hotels i...
Luxury Tier Hot...
SkyNest Holiday...
Premium Family ...


In [55]:
reranked = rerank(question, chunks)

[2, 4, 1, 8, 7, 3, 5, 6, 9, 10]


In [56]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

Luxury Hotels i...
Luxury Tier Hot...
Introduction to...
Luxury Tier Hot...
Luxury Hotels i...
Premium Tier Ho...
Premium Hotels ...
Hotels in Londo...
SkyNest Holiday...
Premium Family ...


In [67]:
question = "What should do if US visa is previously refused?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "US" in c.page_content.lower():
        print(index)

In [68]:
reranked = rerank(question, chunks)

[1, 5, 11, 17, 7, 13, 19, 12, 15, 6, 16, 14, 3, 2, 4, 8, 9, 10, 18, 20]


In [70]:
for index, c in enumerate(reranked):
    if "US" in c.page_content.lower():
        print(index)

In [71]:
reranked[0].page_content

'Application Process and Important Considerations\n\nThis segment details the necessary steps and considerations for applying for a US B1/B2 visa, including the DS-160 online form, interview requirements, reasons for common visa refusals, and the importance of demonstrating strong ties to India.\n\n## Important Notes\n\n- DS-160 online application must be completed first\n- Consular interview is mandatory for first-time applicants\n- ESTA not available to Indian passport holders — B1/B2 visa required\n- Common reasons for refusal: inability to demonstrate intent to return to India; insufficient financial evidence\n- Strong ties to India (family, job, property) significantly improve approval odds\n- US Customs and Border Protection (CBP) determines final entry at the port — visa is not a guaranteed entry\n- SkyNest can provide interview preparation guidance and document checklist'

In [72]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [73]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly travel assistant representing Skynest.
You are chatting with a user about Skynest's travel services including flights, hotels, car rentals, vacation packages, destinations, and visa policies.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [74]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [75]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about Skynest, a travel company.
You are about to look up information in a Knowledge Base to answer the user's question.
The Knowledge Base contains information about Skynest's flights, hotels, car rentals, packages, destinations, and visa policies.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [76]:
rewrite_query("Which hotel has the Aquaventure Waterpark?", [])

'Which hotel features Aquaventure Waterpark?'

In [77]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [78]:
answer_question("Which hotel has the Aquaventure Waterpark?", [])

Which hotel features the Aquaventure Waterpark?
[2, 5, 8, 1, 12, 7, 3, 6, 11, 4, 10, 13, 17, 16, 18, 14, 9, 15, 20, 19]


('The hotel with the Aquaventure Waterpark is Atlantis The Palm in Dubai. It features unlimited access to the waterpark, along with other attractions like the Lost Chambers Aquarium and Dolphin Bay.',
 [Result(page_content='Luxury Hotels in Singapore — SkyNest Partners\n\nThe luxury tier includes top-tier hotels like Marina Bay Sands, The Fullerton Hotel, and Capella Singapore, each offering premium amenities, distinctive vibes, and SkyNest perks.\n\n## Luxury Tier — $300+/night\n\n### Marina Bay Sands ⭐⭐⭐⭐⭐\n- **Location**: Marina Bay, 20 km from SIN\n- **Price**: $450/night (Deluxe) | $780/night (Club Room) | $2,200/night (Chairman Suite)\n- **Vibe**: THE Singapore icon. Infinity pool on the 57th floor (hotel guests only), 3 towers connected by a sky park. Sands SkyPark is the must-see city view.\n- **Facilities**: Infinity pool (57th floor), CÉ LA VI bar/restaurant, 20+ restaurants including Waku Ghin, casino, The Shoppes mall\n- **SkyNest Perk**: Pool access lounge pass + priority 

In [79]:
answer_question("What should do if US visa is previously refused?", [])

What steps to take after a US visa refusal?
[1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


("If your US visa was previously refused, you should carefully review the reasons for the refusal, which are typically provided in the refusal notice. Common reasons include inability to demonstrate strong ties to your home country or insufficient financial evidence.\n\nNext steps:\n- Ensure you address the reasons for refusal before reapplying.\n- You may need to strengthen your supporting documents, such as showing stronger ties to your home country or providing clearer evidence of your financial situation.\n- Schedule a new appointment for a consular interview, if required, and prepare thoroughly. SkyNest offers interview preparation guidance and a document checklist to help you succeed.\n\nRemember, if you've been refused a US visa or had ESTA denied, you usually need to apply for the B1/B2 visa directly at the US embassy or consulate, as ESTA is not available for your nationality or situation.\n\nFor personalized advice and support throughout the process, you can contact SkyNest’s